In [ ]:
import gspread
print("gspread is working!")

# Data Analytics
## Database Type: Google.sheets
### Database : Adventure Works  

```mermaid
flowchart LR
1@{ shape: circle, label: "connect google base-sheet" }
--> 2@{ shape: doc, label: "get information of all sheets" }
--> 3@{ shape: procs, label: "Use Link connect sheet"}
--> 4@{ shape: lin-cyl, label: "Download Records"}
--> 5@{ shape: diam, label: "Check for next Link"}
5 -- Yes --> 3
5 -- No --> 6@{ shape: dbl-circ, label: "End Process"}
```

In [ ]:
import urllib.request
# This script fetches data from a specified URL and prints the content.
# Download the all sheets information locally
url = "https://docs.google.com/spreadsheets/d/e/2PACX-1vQO7atdyPm1anKXSql2oz74C3To18tkzYRIPqhm9YqWX1w3nm73sL8lpybaykBO2g7IB7cvIpQ8W9_u/pub?gid=0&single=true&output=csv"
allsheetsInformation = "AllSheets_Link.csv"

# First, retrieve the data and save it to the local file.
urllib.request.urlretrieve(url, allsheetsInformation)

# Now, open the local file and print its content.
with open(allsheetsInformation, 'r', encoding='utf-8') as response:
    content = response.read()
    
# print(content)

### This flow:
```mermaid
flowchart LR
01[reader] --> 02[read record one]
02 --> 03[go to next record] --> 04{is it more records}
04 -- Yes --> 01
04 -- No --> 05[End]
```

In [ ]:
import csv

with open(allsheetsInformation, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    sheet_info_list = list(reader)

# Example: print each sheet's DataFrame name and link
for sheet in sheet_info_list:
    print(f"{sheet['Data_Frame_Name']} → {sheet['Link']}")

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("GoogleSheetsIngestion") \
    .getOrCreate()

metadata_path = "AllSheets_Link.csv"
metadata_df = spark.read \
    .option("header", True)\
    .option("nullValue", "NULL") \
    .csv(metadata_path)

# metadata_df.show()

In [ ]:
sheet_info = metadata_df.select("Data_Frame_Name", "Link").collect()

for row in sheet_info:
    df_name = row["Data_Frame_Name"]
    csv_url = row["Link"]
    local_path = f"/tmp/{df_name}.csv"  # or any writable path

    urllib.request.urlretrieve(csv_url, local_path)

#### This Flow:

```mermaid
flowchart TB
    subgraph "google links"
    links
    end
    subgraph "downloaded csv"
    links-->CSV1
    links-->CSV2
    links-->CSV3
    end
    subgraph three
    CSV1-->dataFrame1
    CSV2-->dataFrame2
    CSV3-->dataFrame3
    end
```

In [ ]:
dataframes = {}

for row in sheet_info:
    df_name = row["Data_Frame_Name"]
    local_path = f"/tmp/{df_name}.csv"

    df = spark.read \
        .option("header", True) \
        .option("nullValue", "NULL") \
        .option("inferSchema", True) \
        .csv(local_path)

    dataframes[df_name] = df

# print(dataframes.keys())

In [ ]:
Department = dataframes.get("HumanResources_Department").alias("Department")
Employee = dataframes.get("HumanResources_Employee").alias("Employee")
EmployeeDepartmentHistory = dataframes.get("HumanResources_EmployeeDepartmentHistory").alias("EmployeeDepartmentHistory")
EmployeePayHistory = dataframes.get("HumanResources_EmployeePayHistory").alias("EmployeePayHistory")
JobCandidate = dataframes.get("HumanResources_JobCandidate").alias("JobCandidate")
Shift = dataframes.get("HumanResources_Shift").alias("Shift")
Address = dataframes.get("Person_Address").alias("Address")
AddressType = dataframes.get("Person_AddressType").alias("AddressType")
BusinessEntity = dataframes.get("Person_BusinessEntity").alias("BusinessEntity")
BusinessEntityAddress = dataframes.get("Person_BusinessEntityAddress").alias("BusinessEntityAddress")
BusinessEntityContact = dataframes.get("Person_BusinessEntityContact").alias("BusinessEntityContact")
ContactType = dataframes.get("Person_ContactType").alias("ContactType")
CountryRegion = dataframes.get("Person_CountryRegion").alias("CountryRegion")
EmailAddress = dataframes.get("Person_EmailAddress").alias("EmailAddress")
Password = dataframes.get("Person_Password").alias("Password")
Person = dataframes.get("Person_Person").alias("Person")
PersonPhone = dataframes.get("Person_PersonPhone").alias("PersonPhone")
StateProvince = dataframes.get("Person_StateProvince").alias("StateProvince")
BillOfMaterials = dataframes.get("Production_BillOfMaterials").alias("BillOfMaterials")
Culture = dataframes.get("Production_Culture").alias("Culture")
Location = dataframes.get("Production_Location").alias("Location")
Product = dataframes.get("Production_Product").alias("Product")
ProductCategory = dataframes.get("Production_ProductCategory").alias("ProductCategory")
ProductCostHistory = dataframes.get("Production_ProductCostHistory").alias("ProductCostHistory")
ProductDescription = dataframes.get("Production_ProductDescription").alias("ProductDescription")
ProductInventory = dataframes.get("Production_ProductInventory").alias("ProductInventory")
ProductSubcategory = dataframes.get("Production_ProductSubcategory").alias("ProductSubcategory")
TransactionHistory = dataframes.get("Production_TransactionHistory").alias("TransactionHistory")
UnitMeasure = dataframes.get("Production_UnitMeasure").alias("UnitMeasure")
WorkOrder = dataframes.get("Production_WorkOrder").alias("WorkOrder")
WorkOrderRouting = dataframes.get("Production_WorkOrderRouting").alias("WorkOrderRouting")
ProductVendor = dataframes.get("Purchasing_ProductVendor").alias("ProductVendor")
PurchaseOrderDetail = dataframes.get("Purchasing_PurchaseOrderDetail").alias("PurchaseOrderDetail")
PurchaseOrderHeader = dataframes.get("Purchasing_PurchaseOrderHeader").alias("PurchaseOrderHeader")
ShipMethod = dataframes.get("Purchasing_ShipMethod").alias("ShipMethod")
Vendor = dataframes.get("Purchasing_Vendor").alias("Vendor")
CountryRegionCurrency = dataframes.get("Sales_CountryRegionCurrency").alias("CountryRegionCurrency")
CreditCard = dataframes.get("Sales_CreditCard").alias("CreditCard")
Currency = dataframes.get("Sales_Currency").alias("Currency")
CurrencyRate = dataframes.get("Sales_CurrencyRate").alias("CurrencyRate")
Customer = dataframes.get("Sales_Customer").alias("Customer")
PersonCreditCard = dataframes.get("Sales_PersonCreditCard").alias("PersonCreditCard")
SalesOrderDetail = dataframes.get("Sales_SalesOrderDetail").alias("SalesOrderDetail")
SalesOrderHeader = dataframes.get("Sales_SalesOrderHeader").alias("SalesOrderHeader")
SalesOrderHeaderSalesReason = dataframes.get("Sales_SalesOrderHeaderSalesReason").alias("SalesOrderHeaderSalesReason")
SalesPerson = dataframes.get("Sales_SalesPerson").alias("SalesPerson")
SalesTaxRate = dataframes.get("Sales_SalesTaxRate").alias("SalesTaxRate")
SalesTerritory = dataframes.get("Sales_SalesTerritory").alias("SalesTerritory")
SalesTerritoryHistory = dataframes.get("Sales_SalesTerritoryHistory").alias("SalesTerritoryHistory")
ShoppingCartItem = dataframes.get("Sales_ShoppingCartItem").alias("ShoppingCartItem")
SpecialOffer = dataframes.get("Sales_SpecialOffer").alias("SpecialOffer")
SpecialOfferProduct = dataframes.get("Sales_SpecialOfferProduct").alias("SpecialOfferProduct")
Store = dataframes.get("Sales_Store").alias("Store")

##### Join person with person-password and apply filter:
---
```mermaid
erDiagram
direction LR
Person_Person ||--|{ Person_Password : "must have at least one password"
```
---
```mermaid
flowchart LR
    Person_Person e1@--> join_dataFrame
    e1@{ animation: fast }
    Person_Password e2@--> join_dataFrame
    e2@{ animation: fast }
    join_dataFrame e3@-- "filter name=Samantha" --> output([resultset])
    e3@{ animation: fast }
```

In [ ]:
from pyspark.sql.functions import col, count


joined_df = Person.join(
    Password,
    Person["BusinessEntityID"] == Password["BusinessEntityID"],
    how="inner"  # or "left", "right", "outer", "cross"
)

Samantha_joined_df = joined_df.select("person.BusinessEntityID", "person.FirstName", "password.PasswordHash").filter(col("FirstName")=="Samantha")

# Samantha_joined_df.show(truncate=False)

#### Find those name who has multiple passwords:

In [ ]:
# Group by FirstName and count PasswordHash
agg_df = joined_df.groupBy("person.FirstName") \
    .agg(count("password.PasswordHash").alias("PasswordCount"))

# Apply HAVING clause: count > 1
filtered_df = agg_df.filter(col("PasswordCount") > 1)

# Show result
# filtered_df.show(truncate=False)

In [ ]:
# EmployeePayHistory.describe().show()

joined_emppay = Employee.join(
    EmployeePayHistory,
    Employee["BusinessEntityID"] == EmployeePayHistory["BusinessEntityID"],
    how="inner"  # or "left", "right", "outer", "cross"
)

empdetail = Person.join(
    joined_emppay,
    joined_emppay["Employee.BusinessEntityID"] == Person["Person.BusinessEntityID"],
    how="inner"  # or "left", "right", "outer", "cross"
)

# empdetail.show(5, truncate=False)

##### Converted this SQL into Spark-Code
```sql
SELECT Top 10
    e.[BusinessEntityID]
    ,p.[Title]
    ,p.[FirstName]
    ,p.[MiddleName]
    ,p.[LastName]
    ,p.[Suffix]
    ,e.[JobTitle] 
    ,pp.[PhoneNumber]
    ,pnt.[Name] AS [PhoneNumberType]
    ,ea.[EmailAddress]
    ,p.[EmailPromotion]
    ,a.[AddressLine1]
    ,a.[AddressLine2]
    ,a.[City]
    ,sp.[Name] AS [StateProvinceName]
    ,a.[PostalCode]
    ,cr.[Name] AS [CountryRegionName]
    ,p.[AdditionalContactInfo]
FROM [HumanResources].[Employee] e
	INNER JOIN [Person].[Person] p
	ON p.[BusinessEntityID] = e.[BusinessEntityID]
    INNER JOIN [Person].[BusinessEntityAddress] bea
    ON bea.[BusinessEntityID] = e.[BusinessEntityID]
    INNER JOIN [Person].[Address] a
    ON a.[AddressID] = bea.[AddressID]
    INNER JOIN [Person].[StateProvince] sp
    ON sp.[StateProvinceID] = a.[StateProvinceID]
    INNER JOIN [Person].[CountryRegion] cr
    ON cr.[CountryRegionCode] = sp.[CountryRegionCode]
    LEFT OUTER JOIN [Person].[PersonPhone] pp
    ON pp.BusinessEntityID = p.[BusinessEntityID]
    LEFT OUTER JOIN [Person].[PhoneNumberType] pnt
    ON pp.[PhoneNumberTypeID] = pnt.[PhoneNumberTypeID]
    LEFT OUTER JOIN [Person].[EmailAddress] ea
    ON p.[BusinessEntityID] = ea.[BusinessEntityID];
```

In [ ]:
e   = dataframes.get("HumanResources_Employee").alias("e")
p   = dataframes.get("Person_Person").alias("p")
bea = dataframes.get("Person_BusinessEntityAddress").alias("bea")
a   = dataframes.get("Person_Address").alias("a")
sp  = dataframes.get("Person_StateProvince").alias("sp")
cr  = dataframes.get("Person_CountryRegion").alias("cr")
pp  = dataframes.get("Person_PersonPhone").alias("pp")
pnt = dataframes.get("Person_PhoneNumberType").alias("pnt")
ea  = dataframes.get("Person_EmailAddress").alias("ea")

joined_df = e \
    .join(p,   e["BusinessEntityID"] == p["BusinessEntityID"], "inner") \
    .join(bea, e["BusinessEntityID"] == bea["BusinessEntityID"], "inner") \
    .join(a,   bea["AddressID"] == a["AddressID"], "inner") \
    .join(sp,  a["StateProvinceID"] == sp["StateProvinceID"], "inner") \
    .join(cr,  sp["CountryRegionCode"] == cr["CountryRegionCode"], "inner") \
    .join(pp,  p["BusinessEntityID"] == pp["BusinessEntityID"], "left_outer") \
    .join(pnt, pp["PhoneNumberTypeID"] == pnt["PhoneNumberTypeID"], "left_outer") \
    .join(ea,  p["BusinessEntityID"] == ea["BusinessEntityID"], "left_outer") \
    .select(
        col("e.BusinessEntityID"),
        col("p.Title"),
        col("p.FirstName"),
        col("p.MiddleName"),
        col("p.LastName"),
        col("p.Suffix"),
        col("e.JobTitle"),
        col("pp.PhoneNumber"),
        col("pnt.Name").alias("PhoneNumberType"),
        col("ea.EmailAddress"),
        col("p.EmailPromotion"),
        col("a.AddressLine1"),
        col("a.AddressLine2"),
        col("a.City"),
        col("sp.Name").alias("StateProvinceName"),
        col("a.PostalCode"),
        col("cr.Name").alias("CountryRegionName"),
        col("p.AdditionalContactInfo")
        )
# joined_df.show(5, truncate=False)

##### Converted SQL into spark Code 2
```sql
SELECT
    e.[BusinessEntityID]
    ,p.[Title]
    ,p.[FirstName]
    ,p.[MiddleName]
    ,p.[LastName]
    ,p.[Suffix]
    ,e.[JobTitle]
    ,d.[Name] AS [Department]
    ,d.[GroupName]
    ,edh.[StartDate]
FROM [HumanResources].[Employee] e
	INNER JOIN [Person].[Person] p
	ON p.[BusinessEntityID] = e.[BusinessEntityID]
    INNER JOIN [HumanResources].[EmployeeDepartmentHistory] edh
    ON e.[BusinessEntityID] = edh.[BusinessEntityID]
    INNER JOIN [HumanResources].[Department] d
    ON edh.[DepartmentID] = d.[DepartmentID]
WHERE edh.EndDate IS NULL
```

In [ ]:
from pyspark.sql.functions import lit

e = dataframes.get("HumanResources_Employee").alias("e")
p = dataframes.get("Person_Person").alias("p")
edh = dataframes.get("HumanResources_EmployeeDepartmentHistory").alias("edh")
d = dataframes.get("HumanResources_Department").alias("d")

joined_df = e \
    .join(p, e["BusinessEntityID"] == p["BusinessEntityID"], "inner") \
    .join(edh, e["BusinessEntityID"] == edh["BusinessEntityID"], "inner") \
    .join(d, edh["DepartmentID"] == d["DepartmentID"], "inner") \
    .filter(col("edh.EndDate").isNull()) \
    .filter(lit(True)) \
    .filter(col("edh.EndDate").isNull()) \
    .select(
        col("e.BusinessEntityID"),
        col("p.Title"),
        col("p.FirstName"),
        col("p.MiddleName"),
        col("p.LastName"),
        col("p.Suffix"),
        col("e.JobTitle"),
        col("d.Name").alias("Department"),
        col("d.GroupName"),
        col("edh.StartDate")
    )

# joined_df.show(5, truncate=False)


##### Converted SQL into spark Code 3
---
```sql
SELECT
    p.[BusinessEntityID]
    ,p.[Title]
    ,p.[FirstName]
    ,p.[MiddleName]
    ,p.[LastName]
    ,p.[Suffix]
    ,pp.[PhoneNumber]
	,pnt.[Name] AS [PhoneNumberType]
    ,ea.[EmailAddress]
    ,p.[EmailPromotion]
    ,at.[Name] AS [AddressType]
    ,a.[AddressLine1]
    ,a.[AddressLine2]
    ,a.[City]
    ,[StateProvinceName] = sp.[Name]
    ,a.[PostalCode]
    ,[CountryRegionName] = cr.[Name]
    ,p.[Demographics]
FROM [Person].[Person] p
    INNER JOIN [Person].[BusinessEntityAddress] bea
    ON bea.[BusinessEntityID] = p.[BusinessEntityID]
    INNER JOIN [Person].[Address] a
    ON a.[AddressID] = bea.[AddressID]
    INNER JOIN [Person].[StateProvince] sp
    ON sp.[StateProvinceID] = a.[StateProvinceID]
    INNER JOIN [Person].[CountryRegion] cr
    ON cr.[CountryRegionCode] = sp.[CountryRegionCode]
    INNER JOIN [Person].[AddressType] at
    ON at.[AddressTypeID] = bea.[AddressTypeID]
	INNER JOIN [Sales].[Customer] c
	ON c.[PersonID] = p.[BusinessEntityID]
	LEFT OUTER JOIN [Person].[EmailAddress] ea
	ON ea.[BusinessEntityID] = p.[BusinessEntityID]
	LEFT OUTER JOIN [Person].[PersonPhone] pp
	ON pp.[BusinessEntityID] = p.[BusinessEntityID]
	LEFT OUTER JOIN [Person].[PhoneNumberType] pnt
	ON pnt.[PhoneNumberTypeID] = pp.[PhoneNumberTypeID]
WHERE c.StoreID IS NULL;

```

In [ ]:
p = dataframes.get("Person_Person").alias("p")
bea = dataframes.get("Person_BusinessEntityAddress").alias("bea")
a = dataframes.get("Person_Address").alias("a")
sp = dataframes.get("Person_StateProvince").alias("sp")
cr = dataframes.get("Person_CountryRegion").alias("cr")
at = dataframes.get("Person_AddressType").alias("at")
c = dataframes.get("Sales_Customer").alias("c")
ea = dataframes.get("Person_EmailAddress").alias("ea")
pp = dataframes.get("Person_PersonPhone").alias("pp")
pnt = dataframes.get("Person_PhoneNumberType").alias("pnt")


joined_df = p \
    .join(bea, p["BusinessEntityID"] == bea["BusinessEntityID"], "inner") \
    .join(a, bea["AddressID"] == a["AddressID"], "inner") \
    .join(sp, a["StateProvinceID"] == sp["StateProvinceID"], "inner") \
    .join(cr, sp["CountryRegionCode"] == cr["CountryRegionCode"], "inner") \
    .join(at, bea["AddressTypeID"] == at["AddressTypeID"], "inner") \
    .join(c, p["BusinessEntityID"] == c["PersonID"], "inner") \
    .join(ea, p["BusinessEntityID"] == ea["BusinessEntityID"], "left") \
    .join(pp, p["BusinessEntityID"] == pp["BusinessEntityID"], "left") \
    .join(pnt, pp["PhoneNumberTypeID"] == pnt["PhoneNumberTypeID"], "left") \
    .filter(col("c.StoreID").isNull()) \
    .select(
        col("c.StoreID")
        ,col("p.BusinessEntityID")
        ,col("p.Title")
        ,col("p.FirstName")
        ,col("p.MiddleName")
        ,col("p.LastName")
        ,col("p.Suffix")
        ,col("pp.PhoneNumber")
        ,col("pnt.Name").alias("PhoneNumberType")
        ,col("ea.EmailAddress")
        ,col("p.EmailPromotion")
        ,col("at.Name").alias("AddressType")
        ,col("a.AddressLine1")
        ,col("a.AddressLine2")
        ,col("a.City")
        ,col("sp.Name").alias("StateProvinceName")
        ,col("a.PostalCode")
        ,col("cr.Name").alias("CountryRegionName")
        ,col("p.Demographics")
    )
# joined_df.show(5, truncate=False)

##### Converted SQL into spark Code 4
---
```sql
SELECT
    pvt.[SalesPersonID]
    ,pvt.[FullName]
    ,pvt.[JobTitle]
    ,pvt.[SalesTerritory]
    ,pvt.[2012]
    ,pvt.[2013]
    ,pvt.[2014]
FROM (SELECT
        soh.[SalesPersonID]
        ,p.[FirstName] + ' ' + COALESCE(p.[MiddleName], '') + ' ' + p.[LastName] AS [FullName]
        ,e.[JobTitle]
        ,st.[Name] AS [SalesTerritory]
        ,soh.[SubTotal]
        ,YEAR(DATEADD(m, 6, soh.[OrderDate])) AS [FiscalYear]
    FROM [Sales].[SalesPerson] sp
        INNER JOIN [Sales].[SalesOrderHeader] soh
        ON sp.[BusinessEntityID] = soh.[SalesPersonID]
        INNER JOIN [Sales].[SalesTerritory] st
        ON sp.[TerritoryID] = st.[TerritoryID]
        INNER JOIN [HumanResources].[Employee] e
        ON soh.[SalesPersonID] = e.[BusinessEntityID]
		INNER JOIN [Person].[Person] p
		ON p.[BusinessEntityID] = sp.[BusinessEntityID]
	 ) AS soh
PIVOT
(
    SUM([SubTotal])
    FOR [FiscalYear]
    IN ([2012], [2013], [2014])
) AS pvt
```

In [ ]:
from pyspark.sql import functions as F

sp = dataframes.get("Sales_SalesPerson").alias("sp")
soh = dataframes.get("Sales_SalesOrderHeader").alias("soh")
st = dataframes.get("Sales_SalesTerritory").alias("st")
e = dataframes.get("HumanResources_Employee").alias("e")
p = dataframes.get("Person_Person").alias("p")
joined_df = sp \
    .join(soh, sp["BusinessEntityID"] == soh["SalesPersonID"], "inner") \
    .join(st, sp["TerritoryID"] == st["TerritoryID"], "inner") \
    .join(e, soh["SalesPersonID"] == e["BusinessEntityID"], "inner") \
    .join(p, sp["BusinessEntityID"] == p["BusinessEntityID"], "inner") \
    .select(
        col("soh.SalesPersonID"),
        F.concat(F.col("p.FirstName"),   lit(" "),     F.coalesce(col("p.MiddleName"), lit("")), lit(" "), F.col("p.LastName")).alias("FullName"),
        col("e.JobTitle"),
        col("st.Name").alias("SalesTerritory"),
        col("soh.SubTotal"),
        F.year(F.add_months(F.to_date(col("soh.orderdate"),"M/d/yyyy"), 6)).alias("FiscalYear")
    )
joined_df.show(5, truncate=False)
joined_df.printSchema()

pivot_df = joined_df.groupBy("SalesPersonID", "FullName", "JobTitle", "SalesTerritory") \
             .pivot("FiscalYear", [2012, 2013, 2014]) \
             .agg(F.sum("SubTotal")).sort("SalesPersonID")

# pivot_df.filter(col("SalesPersonID")==284).show(truncate=False)
# pivot_df.show(truncate=False)



In [ ]:
employee_df = dataframes.get("HumanResources_Employee")
# employee_df.printSchema()
# employee_df.filter(col("OrganizationLevel").isNull()).show(20, truncate=False)


level_0_df = employee_df.filter(col("OrganizationLevel").isNull())

current_df = level_0_df
all_levels_df = current_df

for i in range(5):  # max depth or until no new rows
    next_df = employee_df.alias("m").join(
        current_df.alias("e"),
        col("m.OrganizationLevel") == col("e.BusinessEntityID"),
        "inner"
    ).select("e.*")
    all_levels_df = all_levels_df.union(next_df)
    current_df = next_df

# all_levels_df.show(truncate=False)

```sql
declare @employee Table
(
EMPLOYEE_ID varchar(100),	FIRST_NAME varchar(100),	LAST_NAME varchar(100),	EMAIL varchar(100),
PHONE_NUMBER varchar(100),	HIRE_DATE varchar(100),	JOB_ID varchar(100),	SALARY Money,
COMMISSION_PCT varchar(100),	MANAGER_ID varchar(100),	DEPARTMENT_ID varchar(100));

Insert @employee
(EMPLOYEE_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE_NUMBER, HIRE_DATE, JOB_ID, SALARY, COMMISSION_PCT, MANAGER_ID, DEPARTMENT_ID)
Values
('198','Donald','OConnell','DOCONNEL','650.507.9833','21-JUN-07','SH_CLERK',2600,' - ','124','50'),
('199','Douglas','Grant','DGRANT','650.507.9844','13-JAN-08','SH_CLERK',2600,' - ','124','50'),
('200','Jennifer','Whalen','JWHALEN','515.123.4444','17-SEP-03','AD_ASST',4400,' - ','101','10'),
('201','Michael','Hartstein','MHARTSTE','515.123.5555','17-FEB-04','MK_MAN',13000,' - ','100','20'),
('202','Pat','Fay','PFAY','603.123.6666','17-AUG-05','MK_REP',6000,' - ','201','20'),
('203','Susan','Mavris','SMAVRIS','515.123.7777','07-JUN-02','HR_REP',6500,' - ','101','40'),
('204','Hermann','Baer','HBAER','515.123.8888','07-JUN-02','PR_REP',10000,' - ','101','70'),
('205','Shelley','Higgins','SHIGGINS','515.123.8080','07-JUN-02','AC_MGR',12008,' - ','101','110'),
('206','William','Gietz','WGIETZ','515.123.8181','07-JUN-02','AC_ACCOUNT',8300,' - ','205','110'),
('100','Steven','King','SKING','515.123.4567','17-JUN-03','AD_PRES',24000,' - ',NULL,'90'),
('101','Neena','Kochhar','NKOCHHAR','515.123.4568','21-SEP-05','AD_VP',17000,' - ','100','90'),
('102','Lex','De Haan','LDEHAAN','515.123.4569','13-JAN-01','AD_VP',17000,' - ','100','90'),
('103','Alexander','Hunold','AHUNOLD','590.423.4567','03-JAN-06','IT_PROG',9000,' - ','102','60'),
('104','Bruce','Ernst','BERNST','590.423.4568','21-MAY-07','IT_PROG',6000,' - ','103','60'),
('105','David','Austin','DAUSTIN','590.423.4569','25-JUN-05','IT_PROG',4800,' - ','103','60'),
('106','Valli','Pataballa','VPATABAL','590.423.4560','05-FEB-06','IT_PROG',4800,' - ','103','60'),
('107','Diana','Lorentz','DLORENTZ','590.423.5567','07-FEB-07','IT_PROG',4200,' - ','103','60'),
('108','Nancy','Greenberg','NGREENBE','515.124.4569','17-AUG-02','FI_MGR',12008,' - ','101','100'),
('109','Daniel','Faviet','DFAVIET','515.124.4169','16-AUG-02','FI_ACCOUNT',9000,' - ','108','100'),
('110','John','Chen','JCHEN','515.124.4269','28-SEP-05','FI_ACCOUNT',8200,' - ','108','100'),
('111','Ismael','Sciarra','ISCIARRA','515.124.4369','30-SEP-05','FI_ACCOUNT',7700,' - ','108','100'),
('112','Jose Manuel','Urman','JMURMAN','515.124.4469','07-MAR-06','FI_ACCOUNT',7800,' - ','108','100'),
('113','Luis','Popp','LPOPP','515.124.4567','07-DEC-07','FI_ACCOUNT',6900,' - ','108','100'),
('114','Den','Raphaely','DRAPHEAL','515.127.4561','07-DEC-02','PU_MAN',11000,' - ','100','30'),
('115','Alexander','Khoo','AKHOO','515.127.4562','18-MAY-03','PU_CLERK',3100,' - ','114','30'),
('116','Shelli','Baida','SBAIDA','515.127.4563','24-DEC-05','PU_CLERK',2900,' - ','114','30'),
('117','Sigal','Tobias','STOBIAS','515.127.4564','24-JUL-05','PU_CLERK',2800,' - ','114','30'),
('118','Guy','Himuro','GHIMURO','515.127.4565','15-NOV-06','PU_CLERK',2600,' - ','114','30'),
('119','Karen','Colmenares','KCOLMENA','515.127.4566','10-AUG-07','PU_CLERK',2500,' - ','114','30'),
('120','Matthew','Weiss','MWEISS','650.123.1234','18-JUL-04','ST_MAN',8000,' - ','100','50'),
('121','Adam','Fripp','AFRIPP','650.123.2234','10-APR-05','ST_MAN',8200,' - ','100','50'),
('122','Payam','Kaufling','PKAUFLIN','650.123.3234','01-MAY-03','ST_MAN',7900,' - ','100','50'),
('123','Shanta','Vollman','SVOLLMAN','650.123.4234','10-OCT-05','ST_MAN',6500,' - ','100','50'),
('124','Kevin','Mourgos','KMOURGOS','650.123.5234','16-NOV-07','ST_MAN',5800,' - ','100','50'),
('125','Julia','Nayer','JNAYER','650.124.1214','16-JUL-05','ST_CLERK',3200,' - ','120','50'),
('126','Irene','Mikkilineni','IMIKKILI','650.124.1224','28-SEP-06','ST_CLERK',2700,' - ','120','50'),
('127','James','Landry','JLANDRY','650.124.1334','14-JAN-07','ST_CLERK',2400,' - ','120','50'),
('128','Steven','Markle','SMARKLE','650.124.1434','08-MAR-08','ST_CLERK',2200,' - ','120','50'),
('129','Laura','Bissot','LBISSOT','650.124.5234','20-AUG-05','ST_CLERK',3300,' - ','121','50'),
('130','Mozhe','Atkinson','MATKINSO','650.124.6234','30-OCT-05','ST_CLERK',2800,' - ','121','50'),
('131','James','Marlow','JAMRLOW','650.124.7234','16-FEB-05','ST_CLERK',2500,' - ','121','50'),
('132','TJ','Olson','TJOLSON','650.124.8234','10-APR-07','ST_CLERK',2100,' - ','121','50'),
('133','Jason','Mallin','JMALLIN','650.127.1934','14-JUN-04','ST_CLERK',3300,' - ','122','50'),
('134','Michael','Rogers','MROGERS','650.127.1834','26-AUG-06','ST_CLERK',2900,' - ','122','50'),
('135','Ki','Gee','KGEE','650.127.1734','12-DEC-07','ST_CLERK',2400,' - ','122','50'),
('136','Hazel','Philtanker','HPHILTAN','650.127.1634','06-FEB-08','ST_CLERK',2200,' - ','122','50'),
('137','Renske','Ladwig','RLADWIG','650.121.1234','14-JUL-03','ST_CLERK',3600,' - ','123','50'),
('138','Stephen','Stiles','SSTILES','650.121.2034','26-OCT-05','ST_CLERK',3200,' - ','123','50'),
('139','John','Seo','JSEO','650.121.2019','12-FEB-06','ST_CLERK',2700,' - ','123','50'),
('140','Joshua','Patel','JPATEL','650.121.1834','06-APR-06','ST_CLERK',2500,' - ','123','50');

with cteRecursive AS
(	Select	EMPLOYEE_ID, FIRST_NAME, LAST_NAME, EMAIL, 
			PHONE_NUMBER, HIRE_DATE, JOB_ID, SALARY, 
			COMMISSION_PCT, CONVERT(varchar(100), NULL) MANAGER_ID, DEPARTMENT_ID,
			CONVERT(varchar(100), NULL) AS MANAGER_FIRST_NAME, CONVERT(varchar(100), NULL) AS MANAGER_LAST_NAME
	from	@employee
	WHERE	1 = 1
		AND	MANAGER_ID IS NULL
	UNION ALL
	Select	employee.EMPLOYEE_ID,employee.FIRST_NAME,employee.LAST_NAME,employee.EMAIL,
			employee.PHONE_NUMBER,employee.HIRE_DATE,employee.JOB_ID,employee.SALARY,
			employee.COMMISSION_PCT,cteRecursive.EMPLOYEE_ID, employee.DEPARTMENT_ID,
			cteRecursive.FIRST_NAME AS MANAGER_FIRST_NAME, cteRecursive.LAST_NAME AS MANAGER_LAST_NAME
	from	@employee employee INNER JOIN
			cteRecursive
			ON	1 = 1
			AND	employee.MANAGER_ID = cteRecursive.EMPLOYEE_ID
)
Select * from cteRecursive;
```

In [ ]:
data = [
    ("198","Donald","OConnell","DOCONNEL","650.507.9833","21-JUN-07","SH_CLERK",2600," - ","124","50"),
("199","Douglas","Grant","DGRANT","650.507.9844","13-JAN-08","SH_CLERK",2600," - ","124","50"),
("200","Jennifer","Whalen","JWHALEN","515.123.4444","17-SEP-03","AD_ASST",4400," - ","101","10"),
("201","Michael","Hartstein","MHARTSTE","515.123.5555","17-FEB-04","MK_MAN",13000," - ","100","20"),
("202","Pat","Fay","PFAY","603.123.6666","17-AUG-05","MK_REP",6000," - ","201","20"),
("203","Susan","Mavris","SMAVRIS","515.123.7777","07-JUN-02","HR_REP",6500," - ","101","40"),
("204","Hermann","Baer","HBAER","515.123.8888","07-JUN-02","PR_REP",10000," - ","101","70"),
("205","Shelley","Higgins","SHIGGINS","515.123.8080","07-JUN-02","AC_MGR",12008," - ","101","110"),
("206","William","Gietz","WGIETZ","515.123.8181","07-JUN-02","AC_ACCOUNT",8300," - ","205","110"),
("100","Steven","King","SKING","515.123.4567","17-JUN-03","AD_PRES",24000," - ","NULL","90"),
("101","Neena","Kochhar","NKOCHHAR","515.123.4568","21-SEP-05","AD_VP",17000," - ","100","90"),
("102","Lex","De Haan","LDEHAAN","515.123.4569","13-JAN-01","AD_VP",17000," - ","100","90"),
("103","Alexander","Hunold","AHUNOLD","590.423.4567","03-JAN-06","IT_PROG",9000," - ","102","60"),
("104","Bruce","Ernst","BERNST","590.423.4568","21-MAY-07","IT_PROG",6000," - ","103","60"),
("105","David","Austin","DAUSTIN","590.423.4569","25-JUN-05","IT_PROG",4800," - ","103","60"),
("106","Valli","Pataballa","VPATABAL","590.423.4560","05-FEB-06","IT_PROG",4800," - ","103","60"),
("107","Diana","Lorentz","DLORENTZ","590.423.5567","07-FEB-07","IT_PROG",4200," - ","103","60"),
("108","Nancy","Greenberg","NGREENBE","515.124.4569","17-AUG-02","FI_MGR",12008," - ","101","100"),
("109","Daniel","Faviet","DFAVIET","515.124.4169","16-AUG-02","FI_ACCOUNT",9000," - ","108","100"),
("110","John","Chen","JCHEN","515.124.4269","28-SEP-05","FI_ACCOUNT",8200," - ","108","100"),
("111","Ismael","Sciarra","ISCIARRA","515.124.4369","30-SEP-05","FI_ACCOUNT",7700," - ","108","100"),
("112","Jose Manuel","Urman","JMURMAN","515.124.4469","07-MAR-06","FI_ACCOUNT",7800," - ","108","100"),
("113","Luis","Popp","LPOPP","515.124.4567","07-DEC-07","FI_ACCOUNT",6900," - ","108","100"),
("114","Den","Raphaely","DRAPHEAL","515.127.4561","07-DEC-02","PU_MAN",11000," - ","100","30"),
("115","Alexander","Khoo","AKHOO","515.127.4562","18-MAY-03","PU_CLERK",3100," - ","114","30"),
("116","Shelli","Baida","SBAIDA","515.127.4563","24-DEC-05","PU_CLERK",2900," - ","114","30"),
("117","Sigal","Tobias","STOBIAS","515.127.4564","24-JUL-05","PU_CLERK",2800," - ","114","30"),
("118","Guy","Himuro","GHIMURO","515.127.4565","15-NOV-06","PU_CLERK",2600," - ","114","30"),
("119","Karen","Colmenares","KCOLMENA","515.127.4566","10-AUG-07","PU_CLERK",2500," - ","114","30"),
("120","Matthew","Weiss","MWEISS","650.123.1234","18-JUL-04","ST_MAN",8000," - ","100","50"),
("121","Adam","Fripp","AFRIPP","650.123.2234","10-APR-05","ST_MAN",8200," - ","100","50"),
("122","Payam","Kaufling","PKAUFLIN","650.123.3234","01-MAY-03","ST_MAN",7900," - ","100","50"),
("123","Shanta","Vollman","SVOLLMAN","650.123.4234","10-OCT-05","ST_MAN",6500," - ","100","50"),
("124","Kevin","Mourgos","KMOURGOS","650.123.5234","16-NOV-07","ST_MAN",5800," - ","100","50"),
("125","Julia","Nayer","JNAYER","650.124.1214","16-JUL-05","ST_CLERK",3200," - ","120","50"),
("126","Irene","Mikkilineni","IMIKKILI","650.124.1224","28-SEP-06","ST_CLERK",2700," - ","120","50"),
("127","James","Landry","JLANDRY","650.124.1334","14-JAN-07","ST_CLERK",2400," - ","120","50"),
("128","Steven","Markle","SMARKLE","650.124.1434","08-MAR-08","ST_CLERK",2200," - ","120","50"),
("129","Laura","Bissot","LBISSOT","650.124.5234","20-AUG-05","ST_CLERK",3300," - ","121","50"),
("130","Mozhe","Atkinson","MATKINSO","650.124.6234","30-OCT-05","ST_CLERK",2800," - ","121","50"),
("131","James","Marlow","JAMRLOW","650.124.7234","16-FEB-05","ST_CLERK",2500," - ","121","50"),
("132","TJ","Olson","TJOLSON","650.124.8234","10-APR-07","ST_CLERK",2100," - ","121","50"),
("133","Jason","Mallin","JMALLIN","650.127.1934","14-JUN-04","ST_CLERK",3300," - ","122","50"),
("134","Michael","Rogers","MROGERS","650.127.1834","26-AUG-06","ST_CLERK",2900," - ","122","50"),
("135","Ki","Gee","KGEE","650.127.1734","12-DEC-07","ST_CLERK",2400," - ","122","50"),
("136","Hazel","Philtanker","HPHILTAN","650.127.1634","06-FEB-08","ST_CLERK",2200," - ","122","50"),
("137","Renske","Ladwig","RLADWIG","650.121.1234","14-JUL-03","ST_CLERK",3600," - ","123","50"),
("138","Stephen","Stiles","SSTILES","650.121.2034","26-OCT-05","ST_CLERK",3200," - ","123","50"),
("139","John","Seo","JSEO","650.121.2019","12-FEB-06","ST_CLERK",2700," - ","123","50"),
("140","Joshua","Patel","JPATEL","650.121.1834","06-APR-06","ST_CLERK",2500," - ","123","50")
]
columns = ["EMPLOYEE_ID","FIRST_NAME","LAST_NAME","EMAIL","PHONE_NUMBER","HIRE_DATE","JOB_ID","SALARY","COMMISSION_PCT","MANAGER_ID","DEPARTMENT_ID"]

df_employee = spark.createDataFrame(data, columns)


# df_employee.printSchema()
# df_employee.show(20, truncate=False)


# df_employee.filter(col("MANAGER_ID")=="NULL").show(20, truncate=False)


level_0_df = df_employee.filter(col("MANAGER_ID") == "NULL")

current_df = level_0_df
all_levels_df = current_df.withColumn("MANAGER_NAME", col("FIRST_NAME"))  # Top-level managers manage themselves

for i in range(5):  # max depth
    next_df = df_employee.alias("e").join(
        current_df.alias("m"),
        col("e.MANAGER_ID") == col("m.EMPLOYEE_ID"),
        "inner"
    ).select(
        col("e.*"),
        col("m.FIRST_NAME").alias("MANAGER_NAME")
    )

    all_levels_df = all_levels_df.union(next_df)
    current_df = next_df


# all_levels_df.select(
#     col("EMPLOYEE_ID"),
#     col("FIRST_NAME"),
#     col("LAST_NAME"),
#     col("MANAGER_ID"),
#     col("MANAGER_NAME"),
#     col("DEPARTMENT_ID")
# ).show(truncate=False)


In [ ]:
from pyspark.sql import functions as F

soh = dataframes.get("Sales_SalesOrderHeader").alias("soh")
c = dataframes.get("Sales_Customer").alias("c")
p = dataframes.get("Person_Person").alias("p")
sod = dataframes.get("Sales_SalesOrderDetail").alias("sod")
prd = dataframes.get("Production_Product").alias("prd")
# prd.printSchema()
# sod.printSchema()
# soh.printSchema()
c.printSchema()

joined_df = soh.join(
    c,
    soh["CustomerID"] == c["CustomerID"],
    how="inner"
) \
.join(
    p, c["PersonID"] == p["BusinessEntityID"],
    how="inner"
) \
.join(
    sod, soh["SalesOrderID"] == sod["SalesOrderID"],
    how="inner"
) \
.join(
    prd, sod["ProductID"] == prd["ProductID"],
    how="inner"
) \
.select(
    F.col("soh.SalesOrderID"),
    F.col("soh.OrderDate"),
    F.col("soh.DueDate"),
    F.col("soh.ShipDate"),
    F.col("soh.Status"),
    F.col("soh.OnlineOrderFlag"),
    F.col("soh.SalesOrderNumber"),
    F.col("soh.PurchaseOrderNumber"),
    F.col("soh.SubTotal"),
    F.col("soh.TaxAmt"),
    F.col("soh.Freight"),
    F.col("soh.TotalDue"),
    F.col("soh.Comment"),
    F.col("c.CustomerID"),
    F.col("p.firstName"),
    F.col("p.lastName"),
    F.col("c.AccountNumber"),
    F.col("prd.name").alias("ProductName"),
    F.col("sod.OrderQty"),
    F.col("sod.UnitPrice"),
    F.col("sod.LineTotal"),
    F.month(F.to_date(col("OrderDate"),"M/d/yyyy"))
)

# joined_df.show(5, truncate=False)
mayFilter = joined_df.filter((F.year(F.to_date(col("OrderDate"),"M/d/yyyy")) == 2011) & (F.month(F.to_date(col("OrderDate"),"M/d/yyyy")) == 5) & (col("AccountNumber") == "AW00029825"))




In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, rank, dense_rank
windowSpec = Window.partitionBy("CustomerID").orderBy("OrderQty")

joined_df_numbered = joined_df.withColumn("row_num", row_number().over(windowSpec))

# joined_df_numbered.filter(col("row_num") <= 10).show(truncate=False)


In [ ]:
# mayFilter.groupBy("CustomerID", "firstName", "lastName", "AccountNumber") \
#          .agg(
#              F.sum("OrderQty"),
#              F.sum("UnitPrice"),
#              F.sum("LineTotal")
#          ) \
#          .show()


```sql
SELECT CustomerID, SUM(LineTotal) AS TotalSpent
FROM Sales
WHERE OrderDate >= '2022-01-01'
GROUP BY CustomerID
ORDER BY TotalSpent DESC
```

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window

#Filter by OrderDate and group by CustomerID

SalesOrderHeader_allCustID = SalesOrderHeader.filter((F.year(F.to_date(col("OrderDate"),"M/d/yyyy")) == 2011) & (F.month(F.to_date(col("OrderDate"),"M/d/yyyy")) == 5)) \
    .groupBy("CustomerID") \
    .agg(F.sum("TotalDue").alias("TotalSpent")) \
    .orderBy(F.col("TotalSpent").desc()) 

# Apply HAVING logic (filter after aggregation)
SalesOrderHeader_TotalSepnd = SalesOrderHeader_allCustID.filter(col("TotalSpent") >= 30000)
# SalesOrderHeader_TotalSepnd.show(truncate=False)


# Add window function: SUM(LineTotal) OVER()

window_spec = F.sum("TotalSpent").over(Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing))
final_df = SalesOrderHeader_TotalSepnd.withColumn("Total_LineTotal", window_spec)
# final_df.show()


#Sort
final_df.orderBy(F.col("TotalSpent").desc()).show(truncate=False)


# <center> End of Script </center>

In [ ]:
spark.stop()